In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2007_2008"
    / "Examination"
)

xpt_file = DATA_DIR / "BMX_E.xpt"
csv_file = DATA_DIR / "BMX_E_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Keep untouched source for final fidelity comparison
source_df = df.copy()


print("=" * 80)
print("NHANES 2007-2008 BMX_E XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "BMDSTATS",
    "BMXWT",
    "BMIWT",
    "BMXRECUM",
    "BMIRECUM",
    "BMXHEAD",
    "BMIHEAD",
    "BMXHT",
    "BMIHT",
    "BMXBMI",
    "BMXLEG",
    "BMILEG",
    "BMXARML",
    "BMIARML",
    "BMXARMC",
    "BMIARMC",
    "BMXWAIST",
    "BMIWAIST",
    "BMXTRI",
    "BMITRI",
    "BMXSUB",
    "BMISUB"
]


if df.shape != (9762, 23):

    raise ValueError(
        f"Unexpected BMX_E shape: {df.shape}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "BMX_E columns/order differ.\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if df.columns.duplicated().any():

    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: BMX_E structure = "
    "9,762 rows x 23 columns."
)


# ============================================================
# 4. DEFINE MEASUREMENT VS INTEGER/CODE FIELDS
# ============================================================

#
# CRITICAL:
#
# These are actual body measurements.
#
# KEEP THEM AS SOURCE-DECODED FLOATS.
#
# Examples:
#
# BMXWT   75.4  -> 75.4
# BMXHT   168.0 -> 168.0
# BMXBMI  25.3  -> 25.3
#
# DO NOT:
#
# - convert to Int64
# - round()
# - use float_format
# - recalculate BMI
# - remove legitimate .0
#

MEASUREMENT_COLUMNS = [
    "BMXWT",
    "BMXRECUM",
    "BMXHEAD",
    "BMXHT",
    "BMXBMI",
    "BMXLEG",
    "BMXARML",
    "BMXARMC",
    "BMXWAIST",
    "BMXTRI",
    "BMXSUB"
]


INTEGER_COLUMNS = [
    col
    for col in df.columns
    if col not in MEASUREMENT_COLUMNS
]


if len(MEASUREMENT_COLUMNS) != 11:

    raise ValueError(
        "Unexpected number of measurement fields."
    )


if len(INTEGER_COLUMNS) != 12:

    raise ValueError(
        "Unexpected number of integer/code fields."
    )


print(
    "\nMeasurement fields preserved AS IS:",
    len(MEASUREMENT_COLUMNS)
)

print(
    "Identifier/status/comment fields:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. SAS/XPORT TINY VALUE CHECK
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns.tolist()


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


print(
    "\n--- ORIGINAL XPT TINY VALUE CHECK ---"
)

print(
    "Tiny artifact occurrences:",
    tiny_total
)


if tiny_total != 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected tiny XPORT values found. "
        "Do NOT automatically replace them with zero."
    )


print(
    "PASS: BMX_E contains NO tiny XPORT artifacts."
)


# ============================================================
# 6. TRUE NUMERIC ZERO CHECK
# ============================================================

true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print(
    "\nTrue numeric zeros in original XPT:",
    true_zero_total
)


if true_zero_total != 0:

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "Unexpected source zero(s) found. "
        "Review before proceeding."
    )


print(
    "PASS: Original BMX_E contains no numeric zeros."
)


# ============================================================
# 7. SEQN VALIDATION
# ============================================================

print(
    "\n--- SEQN CHECK ---"
)


seqn = df["SEQN"]


if int(
    seqn.isna()
    .sum()
) != 0:

    raise ValueError(
        "Missing SEQN."
    )


if int(
    seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN values found."
    )


if seqn.min() != 41475:

    raise ValueError(
        f"Unexpected minimum SEQN: {seqn.min()}"
    )


if seqn.max() != 51623:

    raise ValueError(
        f"Unexpected maximum SEQN: {seqn.max()}"
    )


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=0
    )
):

    raise ValueError(
        "SEQN contains fractional values."
    )


print(
    "PASS: SEQN range 41475-51623."
)

print(
    "PASS: Missing SEQN = 0."
)

print(
    "PASS: Duplicate SEQN = 0."
)


# ============================================================
# 8. FRACTIONAL VALUE CHECK
# ============================================================

#
# Every genuine fractional value should occur ONLY
# in body-measurement fields.
#

fractional_columns = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    count = int(
        fractional.sum()
    )


    if count > 0:

        fractional_columns[col] = count


print(
    "\n--- FRACTIONAL VALUE CHECK ---"
)


for col, count in fractional_columns.items():

    print(
        f"{col:<10}: {count:,}"
    )


if set(
    fractional_columns.keys()
) != set(
    MEASUREMENT_COLUMNS
):

    raise ValueError(
        "Unexpected fractional-variable set.\n"
        f"Expected:\n{MEASUREMENT_COLUMNS}\n"
        f"Actual:\n{list(fractional_columns)}"
    )


print(
    "PASS: Genuine fractional values occur "
    "only in the 11 body-measurement fields."
)


# ============================================================
# 9. VERIFY INTEGER/CODE FIELDS ARE WHOLE NUMBERS
# ============================================================

integer_fraction_failures = {}


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional_count = int(
        (
            ~np.isclose(
                values,
                np.round(values),
                rtol=0,
                atol=1e-12
            )
        ).sum()
    )


    if fractional_count > 0:

        integer_fraction_failures[
            col
        ] = fractional_count


if integer_fraction_failures:

    raise ValueError(
        "STOP: Identifier/status/comment fields "
        "contain fractional values:\n"
        + str(
            integer_fraction_failures
        )
    )


print(
    "PASS: SEQN/status/comment fields "
    "are all whole-number values."
)


# ============================================================
# 10. CDC MEASUREMENT RANGE / VALID COUNT CHECKS
# ============================================================

#
# These are the released CDC ranges and counts.
#

EXPECTED_MEASUREMENTS = {

    "BMXWT": {
        "valid": 9631,
        "missing": 131,
        "min": 3.1,
        "max": 218.2
    },

    "BMXRECUM": {
        "valid": 1194,
        "missing": 8568,
        "min": 49.1,
        "max": 116.6
    },

    "BMXHEAD": {
        "valid": 260,
        "missing": 9502,
        "min": 33.5,
        "max": 47.4
    },

    "BMXHT": {
        "valid": 8873,
        "missing": 889,
        "min": 81.5,
        "max": 203.8
    },

    "BMXBMI": {
        "valid": 8861,
        "missing": 901,
        "min": 12.5,
        "max": 73.43
    },

    "BMXLEG": {
        "valid": 7304,
        "missing": 2458,
        "min": 22.1,
        "max": 55.5
    },

    "BMXARML": {
        "valid": 9276,
        "missing": 486,
        "min": 9.0,
        "max": 47.7
    },

    "BMXARMC": {
        "valid": 9281,
        "missing": 481,
        "min": 9.6,
        "max": 59.4
    },

    "BMXWAIST": {
        "valid": 8538,
        "missing": 1224,
        "min": 37.8,
        "max": 178.2
    },

    "BMXTRI": {
        "valid": 8944,
        "missing": 818,
        "min": 3.0,
        "max": 41.4
    },

    "BMXSUB": {
        "valid": 8213,
        "missing": 1549,
        "min": 3.0,
        "max": 44.0
    }
}


print(
    "\n--- CDC MEASUREMENT CHECKS ---"
)


for col, spec in EXPECTED_MEASUREMENTS.items():

    s = df[col]


    actual_valid = int(
        s.notna()
        .sum()
    )


    actual_missing = int(
        s.isna()
        .sum()
    )


    if actual_valid != spec["valid"]:

        raise ValueError(
            f"{col}: valid count differs.\n"
            f"Expected: {spec['valid']:,}\n"
            f"Actual:   {actual_valid:,}"
        )


    if actual_missing != spec["missing"]:

        raise ValueError(
            f"{col}: missing count differs.\n"
            f"Expected: {spec['missing']:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    if not np.isclose(
        s.min(),
        spec["min"],
        rtol=0,
        atol=1e-12
    ):

        raise ValueError(
            f"{col}: minimum differs.\n"
            f"Expected: {spec['min']}\n"
            f"Actual:   {s.min()}"
        )


    if not np.isclose(
        s.max(),
        spec["max"],
        rtol=0,
        atol=1e-12
    ):

        raise ValueError(
            f"{col}: maximum differs.\n"
            f"Expected: {spec['max']}\n"
            f"Actual:   {s.max()}"
        )


    print(
        f"PASS: {col:<10} "
        f"valid={actual_valid:,} "
        f"missing={actual_missing:,} "
        f"range={s.min()}-{s.max()}"
    )


print(
    "PASS: All measurement ranges/counts match CDC."
)


# ============================================================
# 11. CDC STATUS / COMMENT CODE CHECKS
# ============================================================

EXPECTED_CODE_COUNTS = {

    "BMDSTATS": {
        "counts": {
            1: 8088,
            2: 241,
            3: 1322,
            4: 111
        },
        "missing": 0
    },

    "BMIWT": {
        "counts": {
            1: 28,
            3: 292,
            4: 19
        },
        "missing": 9423
    },

    "BMIRECUM": {
        "counts": {
            1: 17,
            3: 1
        },
        "missing": 9744
    },

    "BMIHEAD": {
        "counts": {},
        "missing": 9762
    },

    "BMIHT": {
        "counts": {
            1: 50,
            3: 259
        },
        "missing": 9453
    },

    "BMILEG": {
        "counts": {
            1: 324
        },
        "missing": 9438
    },

    "BMIARML": {
        "counts": {
            1: 299
        },
        "missing": 9463
    },

    "BMIARMC": {
        "counts": {
            1: 302
        },
        "missing": 9460
    },

    "BMIWAIST": {
        "counts": {
            1: 369
        },
        "missing": 9393
    },

    "BMITRI": {
        "counts": {
            1: 442,
            2: 199
        },
        "missing": 9121
    },

    "BMISUB": {
        "counts": {
            1: 1232,
            2: 140
        },
        "missing": 8390
    }
}


print(
    "\n--- CDC STATUS / COMMENT CHECKS ---"
)


for col, spec in EXPECTED_CODE_COUNTS.items():

    actual_counts = (
        df[col]
        .value_counts(
            dropna=True
        )
        .sort_index()
        .to_dict()
    )


    actual_counts = {
        int(k): int(v)
        for k, v in actual_counts.items()
    }


    actual_missing = int(
        df[col]
        .isna()
        .sum()
    )


    if actual_counts != spec["counts"]:

        raise ValueError(
            f"{col}: CDC frequency mismatch.\n"
            f"Expected: {spec['counts']}\n"
            f"Actual:   {actual_counts}"
        )


    if actual_missing != spec["missing"]:

        raise ValueError(
            f"{col}: missing count mismatch.\n"
            f"Expected: {spec['missing']:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {col:<10} "
        f"valid={sum(actual_counts.values()):,} "
        f"missing={actual_missing:,}"
    )


print(
    "PASS: Status/comment frequencies match CDC."
)


# ============================================================
# 12. EXPECTED DATASET FOR FIDELITY
# ============================================================

#
# IMPORTANT:
#
# BMX_E requires NO numeric corrections.
#
# There are:
#
# tiny -> zero corrections = 0
#
# Therefore the expected numeric data is the
# original released XPT exactly.
#

expected_df = source_df.copy()


# ============================================================
# 13. CLEAN ONLY SEQN / STATUS / COMMENT FIELDS
# ============================================================

#
# Safe:
#
# SEQN
# 41475.0 -> 41475
#
# BMDSTATS
# 1.0 -> 1
#
# BMIWT
# 3.0 -> 3
#
#
# NOT SAFE:
#
# BMXWT
# 70.0 -> 70
#
# BMXHT
# 170.0 -> 170
#
# BMXBMI
# 25.0 -> 25
#
# Measurements stay float.
#

for col in INTEGER_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: SEQN/status/comment fields "
    "converted to clean integer formatting."
)

print(
    "PASS: Body measurement fields "
    "left as source-decoded floats."
)


# ============================================================
# 14. PRE-EXPORT EXACT NUMERIC FIDELITY
# ============================================================

print(
    "\n--- PRE-EXPORT NUMERIC FIDELITY ---"
)


pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    if col in MEASUREMENT_COLUMNS:

        current_values = (
            df[col]
            .to_numpy()
        )

    else:

        current_values = (
            df[col]
            .astype("Float64")
            .to_numpy(
                dtype=float,
                na_value=np.nan
            )
        )


    if not np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: No underlying BMX_E numeric value changed."
)


# ============================================================
# 15. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO:
#
# round()
# float_format
# recalculation
#
# Missing values -> blank CSV cells.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 16. RE-READ CSV WITH ROUND-TRIP FLOAT PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 17. STRUCTURE AFTER EXPORT
# ============================================================

if csv_df.shape != expected_df.shape:

    raise ValueError(
        "CSV shape differs from XPT.\n"
        f"Expected: {expected_df.shape}\n"
        f"Actual:   {csv_df.shape}"
    )


if csv_df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "\nPASS: CSV structure matches XPT."
)


# ============================================================
# 18. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

numeric_failures = []
total_numeric_differences = 0


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    equal = np.isclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    differences = int(
        (~equal).sum()
    )


    total_numeric_differences += (
        differences
    )


    if differences > 0:

        numeric_failures.append(
            (
                col,
                differences
            )
        )


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)


if numeric_failures:

    raise ValueError(
        "Numeric fidelity failure:\n"
        + str(
            numeric_failures
        )
    )


print(
    "PASS: All 23 CSV columns numerically "
    "match the original XPT exactly."
)


# ============================================================
# 19. MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0
missing_failures = []


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    differences = int(
        (
            expected_missing
            !=
            exported_missing
        ).sum()
    )


    missing_mismatches += (
        differences
    )


    if differences > 0:

        missing_failures.append(
            (
                col,
                differences
            )
        )


print(
    "Missing-position differences:",
    missing_mismatches
)


if missing_failures:

    raise ValueError(
        "Missing positions changed:\n"
        + str(
            missing_failures
        )
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 20. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f)

    raw_rows = list(reader)


if len(raw_rows) != 9762:

    raise ValueError(
        "Unexpected raw CSV row count."
    )


# ============================================================
# 21. STRICT RAW CSV FIDELITY
# ============================================================

#
# Integer/status/comment fields:
#
# str(int(source_value))
#
#
# Measurement fields:
#
# str(float(source_value))
#
# This means:
#
# BMXWT 70.0 stays "70.0"
# BMXWT 70.4 stays "70.4"
#
# BMXBMI 25.0 stays "25.0"
# BMXBMI 25.3 stays "25.3"
#

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in EXPECTED_COLUMNS:

        source_value = (
            expected_df.iloc[i][col]
        )


        if col in MEASUREMENT_COLUMNS:

            expected_text = (
                ""
                if pd.isna(source_value)
                else str(
                    float(source_value)
                )
            )

        else:

            expected_text = (
                ""
                if pd.isna(source_value)
                else str(
                    int(source_value)
                )
            )


        actual_text = row[col]


        if actual_text != expected_text:

            raw_mismatches += 1


            if len(
                raw_examples
            ) < 20:

                raw_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "\nRaw-format differences:",
    raw_mismatches
)


if raw_mismatches != 0:

    print(
        "\nExamples:"
    )


    for example in raw_examples:

        print(
            example
        )


    raise ValueError(
        "BMX_E raw CSV formatting differs."
    )


print(
    "PASS: Raw CSV tokens match "
    "the expected source representation."
)


# ============================================================
# 22. UNWANTED .0 CHECK
#     INTEGER/CODE FIELDS ONLY
# ============================================================

#
# IMPORTANT:
#
# Do NOT globally remove ".0".
#
# ".0" is legitimate in body measurements.
#

unwanted_dot_zero_count = 0
unwanted_dot_zero_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col in INTEGER_COLUMNS:

        value = row[col]


        if (
            value != ""
            and
            value.endswith(".0")
        ):

            unwanted_dot_zero_count += 1


            if len(
                unwanted_dot_zero_examples
            ) < 20:

                unwanted_dot_zero_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nUnwanted integer-field '.0' values:",
    unwanted_dot_zero_count
)


if unwanted_dot_zero_count != 0:

    print(
        unwanted_dot_zero_examples
    )

    raise ValueError(
        "Unnecessary .0 remains in "
        "SEQN/status/comment fields."
    )


print(
    "PASS: No unwanted .0 in integer/code fields."
)


# ============================================================
# 23. STRICT BODY-MEASUREMENT RAW CHECK
# ============================================================

measurement_raw_mismatches = 0
measurement_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in MEASUREMENT_COLUMNS:

        source_value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(source_value)
            else str(
                float(source_value)
            )
        )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            measurement_raw_mismatches += 1


            if len(
                measurement_examples
            ) < 20:

                measurement_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "Body-measurement raw differences:",
    measurement_raw_mismatches
)


if measurement_raw_mismatches != 0:

    print(
        measurement_examples
    )

    raise ValueError(
        "Body measurement decimal representation changed."
    )


print(
    "PASS: Every body measurement is "
    "preserved AS IS."
)


# ============================================================
# 24. LEGITIMATE MEASUREMENT .0 CHECK
# ============================================================

#
# Some measurement values are mathematically whole
# numbers but must remain float-style tokens.
#
# Determine expected count directly from source.
#

print(
    "\n--- LEGITIMATE MEASUREMENT .0 VALUES ---"
)


measurement_dot_zero_failures = {}


for col in MEASUREMENT_COLUMNS:

    expected_count = sum(
        (
            ""
            if pd.isna(value)
            else str(float(value))
        ).endswith(".0")
        for value in expected_df[col]
    )


    actual_count = sum(
        row[col].endswith(".0")
        for row in raw_rows
        if row[col] != ""
    )


    print(
        f"{col:<10}: "
        f"{actual_count:,}"
    )


    if actual_count != expected_count:

        measurement_dot_zero_failures[
            col
        ] = {
            "expected": expected_count,
            "actual": actual_count
        }


if measurement_dot_zero_failures:

    raise ValueError(
        "Legitimate measurement .0 formatting changed:\n"
        + str(
            measurement_dot_zero_failures
        )
    )


print(
    "PASS: Legitimate measurement .0 values preserved."
)


# ============================================================
# 25. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = (
            value.lower()
        )


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 26. FINAL TINY-ARTIFACT TEXT CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


tiny_string_count = (
    csv_text.count(
        "5.397605346934028e-79"
    )
)


print(
    "Tiny artifact strings in CSV:",
    tiny_string_count
)


if tiny_string_count != 0:

    raise ValueError(
        "Tiny XPORT artifact found in CSV."
    )


print(
    "PASS: No tiny artifact in CSV."
)


# ============================================================
# 27. FINAL CDC RANGE CHECK AFTER EXPORT
# ============================================================

for col, spec in EXPECTED_MEASUREMENTS.items():

    s = csv_df[col]


    if int(
        s.notna().sum()
    ) != spec["valid"]:

        raise ValueError(
            f"{col}: valid count changed after export."
        )


    if not np.isclose(
        s.min(),
        spec["min"],
        rtol=0,
        atol=1e-12
    ):

        raise ValueError(
            f"{col}: minimum changed after export."
        )


    if not np.isclose(
        s.max(),
        spec["max"],
        rtol=0,
        atol=1e-12
    ):

        raise ValueError(
            f"{col}: maximum changed after export."
        )


print(
    "\nPASS: Final measurement ranges "
    "still match CDC."
)


# ============================================================
# 28. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("BMX_E CONVERSION COMPLETE")
print("=" * 80)


print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    "41475-51623"
)


print(
    "\nTiny artifacts in source:",
    tiny_total
)

print(
    "Tiny -> zero corrections:",
    0
)

print(
    "True source zeros:",
    true_zero_total
)


print(
    "\nBody measurement fields preserved AS IS:",
    len(MEASUREMENT_COLUMNS)
)


for col in MEASUREMENT_COLUMNS:

    print(
        col
    )


print(
    "\nVariables with genuine fractional values:",
    len(fractional_columns)
)


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)

print(
    "Missing-position differences:",
    missing_mismatches
)

print(
    "Raw-format differences:",
    raw_mismatches
)

print(
    "Body-measurement raw differences:",
    measurement_raw_mismatches
)

print(
    "Unwanted integer .0 values:",
    unwanted_dot_zero_count
)

print(
    "Scientific notation:",
    scientific_count
)


print(
    "\nEXPECTED FINAL STATUS:"
)

print(
    "BMX_E_corrected.csv = "
    "CORRECTED + FIDELITY CHECKED"
)

NHANES 2007-2008 BMX_E XPT -> CSV
Rows: 9,762
Columns: 23
PASS: BMX_E structure = 9,762 rows x 23 columns.

Measurement fields preserved AS IS: 11
Identifier/status/comment fields: 12

--- ORIGINAL XPT TINY VALUE CHECK ---
Tiny artifact occurrences: 0
PASS: BMX_E contains NO tiny XPORT artifacts.

True numeric zeros in original XPT: 0
PASS: Original BMX_E contains no numeric zeros.

--- SEQN CHECK ---
PASS: SEQN range 41475-51623.
PASS: Missing SEQN = 0.
PASS: Duplicate SEQN = 0.

--- FRACTIONAL VALUE CHECK ---
BMXWT     : 8,642
BMXRECUM  : 1,072
BMXHEAD   : 225
BMXHT     : 8,002
BMXBMI    : 8,764
BMXLEG    : 5,835
BMXARML   : 6,210
BMXARMC   : 7,775
BMXWAIST  : 6,963
BMXTRI    : 6,520
BMXSUB    : 6,003
PASS: Genuine fractional values occur only in the 11 body-measurement fields.
PASS: SEQN/status/comment fields are all whole-number values.

--- CDC MEASUREMENT CHECKS ---
PASS: BMXWT      valid=9,631 missing=131 range=3.1-218.2
PASS: BMXRECUM   valid=1,194 missing=8,568 range=49.1-116.